# CNW-Grass gas exchange, 6: with other models

**What you will learn:** what the gas exchange takes from the other models it runs with: the light from Caribu, the
plants from Adel, and the leaf water potential from a water model such as MECHA.

**You need:** tutorial 1 (its last section). **Run time:** about 15 seconds.

## 1. The light: Caribu

Caribu lights the stand every hour: each element absorbs a share of the light above the stand, depending on the
sun's position and on the leaves around it. Its result, each element's absorbed PAR (`PARa`), is the gas exchange's
input of the same name. At noon, the blades by the light they absorb (their share of the stand's blade area, and
their mean net assimilation):

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from openalea.cnwgrass.metafspm.simulation import simulate

noon = simulate(hours=1, start_hour=12)
elements = noon.elements()
blades = elements[elements["organ"] == "blade"]
light = pd.cut(blades["PARa"], [0, 250, 500, 1000, 1500, 2000], include_lowest=True)
classes = blades.groupby(light, observed=True)
pd.DataFrame({"share of the blade area": classes["green_area"].sum() / blades["green_area"].sum(),
              "mean An": classes["An"].mean()}).rename_axis("PAR absorbed (µmol m-2 s-1)").round(2)

,share of the blade area,mean An
PAR absorbed (µmol m-2 s-1),,
"(-0.001, 250.0]",0.20,4.83
"(250.0, 500.0]",0.20,16.51
"(500.0, 1000.0]",0.34,25.13
"(1000.0, 1500.0]",0.20,30.92
"(1500.0, 2000.0]",0.06,31.10


At the same moment, some blades absorb a few tens of µmol m⁻² s⁻¹, shaded by their neighbours, and others more
than 1000, facing the sun. Their photosynthesis follows (tutorial 1's light response): a light model that gave every
leaf the same mean light would miss that the sunlit leaves are saturated and the shaded ones are not.

## 2. The plants: Adel

Adel computes each element's area, width and height as the plants grow. The stand at three stages, at noon (the
leaf area index, LAI, from the green area):

In [2]:
rows = {}
for thermal_time in (400., 600., 800.):
    result = simulate(hours=1, start_hour=12, thermal_time=thermal_time)
    row = result.summary.iloc[0]
    rows[f"{thermal_time:.0f} °Cd"] = {"LAI": result.plants()["green_area"].sum() / result.ground_area,
                                      "PAR absorbed": row["PAR_absorbed"], "net assimilation": row["net_assimilation"],
                                      "transpiration": row["transpiration"]}
pd.DataFrame(rows).T.round(3)

,LAI,PAR absorbed,net assimilation,transpiration
400 °Cd,0.279,233.775,6.544,1.385
600 °Cd,0.594,362.484,10.747,2.105
800 °Cd,0.885,471.466,13.268,2.504


A larger canopy absorbs more light and fixes more CO₂ per m² of ground. The plants also grow during a simulation, by
the degree-days of the air temperature: over a day at 20 °C, 20 °Cd.

## 3. The water: a hydraulic model

The stomatal models `Tuzet` and `hydraulics` close the stomata when the leaves are short of water, as their **leaf
water potential** falls (in MPa; the more negative, the drier). In a complete plant model, a water flow model such as
MECHA computes it from the soil's water and the transpiration; here `simulate` gives every element the same value,
`leaf_water_potential=`.

In [3]:
columns = ["net_assimilation", "transpiration", "stomatal_conductance", "surface_temperature"]
rows = {}
for potential in (-0.1, -0.4, -0.65, -1.0):
    rows[f"{potential} MPa"] = simulate({"stomatal_model": "Tuzet"}, hours=1, start_hour=12,
                                        leaf_water_potential=potential).summary.iloc[0][columns]
pd.DataFrame(rows).T.round(3)

,net_assimilation,transpiration,stomatal_conductance,surface_temperature
-0.1 MPa,11.238,2.250,0.522,23.253
-0.4 MPa,11.135,2.214,0.459,23.483
-0.65 MPa,10.389,2.031,0.268,24.647
-1.0 MPa,6.541,1.475,0.092,28.163


As the leaves dry, their stomata close: they transpire less, fix less CO₂ and warm up. In a coupled model the loop
closes: less transpiration lets the plant's water potential recover. The developer tutorial
`../developer/01_gas_exchange_in_metafspm.ipynb` shows how the gas exchange is coupled to the light and to MECHA.

## Try it yourself

1. With `leaf_water_potential=-0.65`, compare `Tuzet` with `hydraulics`, which also closes the stomata in dry air. Try it
   with `relative_humidity=0.3`.
2. Run the stand for three days (`hours=72`) and plot its net assimilation: does it grow from day to day?